# Notebook 20 — Supervised Fine-Tuning with LoRA and QLoRA

    ## Learning objectives

    - Choose among prompting, RAG, full fine-tuning, LoRA, and QLoRA
- Explain low-rank adapters mathematically and count trainable parameters
- Configure a guarded TRL SFT run with evaluation

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = ['transformers>=4.51,<5', 'datasets>=3.5,<6', 'peft>=0.15', 'trl>=0.16', 'accelerate>=1.6', 'bitsandbytes>=0.45', 'sentencepiece']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if IN_COLAB and not token:
    from google.colab import userdata
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            token = None
        if token:
            break
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub variable. The course also sets its
# descriptive alias because some lesson code uses HUGGINGFACE_TOKEN explicitly.
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGINGFACE_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if True and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("Add an HF_TOKEN secret in Colab, enable notebook access, then rerun this cell.")


## 20.1 What fine-tuning changes

Use RAG to supply changing knowledge; use SFT to teach behavior, format, terminology,
or a task distribution. LoRA freezes base weight \(W\) and learns
\(\Delta W=(\alpha/r)BA\), where rank \(r\) is small. QLoRA stores the frozen base in
low precision while training adapters, reducing weight memory. Neither technique fixes
bad labels, missing evaluation, or an unsuitable base model.


In [ ]:
def lora_params(in_features, out_features, rank):
    return rank * (in_features + out_features)
d = 4096
full = d * d
for rank in [4, 8, 16, 64]:
    adapter = lora_params(d, d, rank)
    print(rank, f"{adapter:,}", f"({adapter/full:.3%} of matrix)")


## 20.2 A reproducible SFT configuration

Evaluate the untouched base model first. Freeze dataset/model revisions. Separate
validation by source or time when duplicates are likely. Inspect rendered chat text.
Monitor both loss and task metrics. Adapter rank, target modules, LR, effective batch,
packing, max length, and loss masking are experimental variables—not boilerplate.


In [ ]:
# Optional GPU training: set True only after `uv sync --extra train`.
RUN_TRAINING = False
if RUN_TRAINING:
    from datasets import load_dataset
    from peft import LoraConfig
    from trl import SFTConfig, SFTTrainer

    dataset = load_dataset("trl-lib/Capybara", split="train[:1000]")
    split = dataset.train_test_split(test_size=0.1, seed=42)
    peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                             bias="none", task_type="CAUSAL_LM")
    args = SFTConfig(
        output_dir="artifacts/qwen-sft-lora", max_length=1024, packing=True,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=2e-4, num_train_epochs=1, gradient_checkpointing=True, fp16=True,
        eval_strategy="steps", eval_steps=50, save_steps=50, report_to="none",
    )
    trainer = SFTTrainer(model="Qwen/Qwen2.5-0.5B-Instruct", args=args,
                         train_dataset=split["train"], eval_dataset=split["test"],
                         peft_config=peft_config)
    trainer.train()
else:
    print("Training skipped. Review configuration, then opt in on a suitable GPU.")


## 20.3 Validate the adapter

Compare base and adapter on frozen prompts with greedy and sampled runs. Check task
quality, general capability retention, safety behavior, latency, and adapter load/merge
behavior. Store adapter weights, base revision, tokenizer, chat template, config, data
fingerprint, code revision, and evaluation report together.


## 20.4 LoRA mechanics and target selection

For a frozen projection \(y=xW^T\), LoRA adds
\((\alpha/r)xA^TB^T\). One factor is commonly initialized randomly and the other to zero,
so the adapter initially leaves model behavior unchanged. Rank controls capacity; alpha
scales the update; dropout regularizes adapter input. Rank and alpha are coupled through the
scaling convention. Adapter parameters are small, but forward/backward through the frozen
base and activations still consume compute/memory.

Attention Q/V targets are a small baseline. Adding K/O and MLP projections increases
capacity and trainable parameters. Module names vary by architecture; print matched modules
and trainable counts rather than trusting a copied pattern. Rank-stabilized LoRA, DoRA, and
adaptive-rank methods alter parameterization, but data and evaluation usually matter more
than chasing variants prematurely.


In [ ]:
# Inspect candidate projection names and calculate adapter coverage without training.
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
hidden = cfg.hidden_size
intermediate = cfg.intermediate_size
rank = 16
matrices = {
    "q_proj": (hidden, hidden), "k_proj": (hidden, hidden),
    "v_proj": (hidden, hidden), "o_proj": (hidden, hidden),
    "gate_proj": (hidden, intermediate), "up_proj": (hidden, intermediate),
    "down_proj": (intermediate, hidden),
}
for name, (inp, out) in matrices.items():
    print(name, f"{rank*(inp+out):,} LoRA params/layer")


## 20.5 QLoRA in detail

QLoRA quantizes the *frozen* base weights, commonly to 4-bit NormalFloat (NF4), while LoRA
adapters train in BF16/FP16. Double quantization compresses quantization constants. Paged
optimizers can handle memory spikes. Dequantization occurs for computation; this is not
equivalent to training 4-bit adapter values. Compute dtype, quantization type, device map,
and hardware kernel support must be explicit.

Quantization reduces base weight memory, but activations, adapter gradients, and optimizer
states remain. Quality can regress for some models/tasks, and merging adapters generally
requires a suitable higher-precision base. `bitsandbytes` is CUDA-oriented; platform support
changes. In Colab, verify the installed library, GPU compute capability, loaded parameter
dtypes, and actual allocated memory. A config flag without inspecting the loaded model is not
proof that QLoRA is active.


In [ ]:
# Reference QLoRA configuration (construction only; use inside the guarded GPU cell).
try:
    from transformers import BitsAndBytesConfig
    import torch
    qlora_quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    print(qlora_quantization)
except Exception as exc:
    print("QLoRA configuration unavailable in this runtime:", type(exc).__name__)


## 20.6 SFT experiment design and deployment

Establish base-model results using the exact inference template. Train on behavior the
prompt/RAG baseline cannot achieve reliably. Use validation sources disjoint from training,
plus a frozen capability/safety suite. Tune LR, effective tokens/update, epochs/tokens, rank,
targets, max length, masking, and data mixture conservatively. Small curated sets can overfit
quickly; more epochs are not automatically useful.

At inference, load base plus adapter dynamically or merge the update. Dynamic adapters enable
multiple tasks but add routing/operational complexity. Merging simplifies serving but produces
a new weight artifact and can interact with quantization. Record base commit, adapter config,
tokenizer/template, library versions, data fingerprint, and eval report. Test that a fresh
process can reconstruct outputs from published artifacts.

**Failure diagnosis:** no behavior change—mask/targets/LR/modules; memorization—duplicates or
too many epochs; broad regression—data narrowness/LR/targets; malformed chat—template/EOS;
OOM—length/activations/precision; slow training—padding, checkpoint recompute, dataloader, or
unsupported kernels.


## 20.7 SFT/adapter reference

| Choice | Typical implication |
|---|---|
| Full fine-tune | Maximum flexibility; gradients/optimizer for all weights |
| LoRA | Frozen base plus small trainable low-rank updates |
| QLoRA | Quantized frozen base plus LoRA adapters |
| Attention-only targets | Small adapter; may limit behavior capacity |
| Attention + MLP | More capacity/parameters and potential regression |
| Merge adapter | Simpler single artifact; loses easy modular routing |

Fine-tuning is appropriate for stable behavior/task distribution, not rapidly changing facts. Compare
prompting and RAG first. A tiny adapter file is inseparable from its exact base model, tokenizer, and
template. Count trainable parameters and inspect matched modules. Validate that optimizer contains only
intended parameters and that ignored labels leave assistant target tokens.

Before publishing: base-versus-adapter paired evals, safety/capability retention, artifact reload in a
fresh environment, license/data documentation, merge/quantization quality checks, inference latency,
and rollback. Treat adapter inputs and outputs as model releases, not miscellaneous experiment files.


## 20.8 Target modules and adapter capacity

LoRA represents a weight update as low-rank factors, but target-module names and coverage differ across architectures. Enumerate matched modules, trainable parameters, ranks, scaling, and dropout; fail if an intended pattern matches nothing. Attention-only adapters may be insufficient for a domain shift that depends on MLP features. Higher rank increases capacity and optimizer state. Compare against prompt-only, full fine-tuning when feasible, and parameter-matched adapter configurations. Save the base revision and adapter configuration together.


In [ ]:
shapes=[("q_proj",4096,4096),("v_proj",4096,4096),("up_proj",4096,11008)]; rank=16
for name,inp,out in shapes: print(name,"full",inp*out,"LoRA",rank*(inp+out),"fraction",rank*(inp+out)/(inp*out))


## 20.9 Assistant-only masking audit

Chat SFT should normally compute loss on assistant tokens while masking padding, system, user, and tool-result spans according to the chosen objective. Character offsets are unsafe after normalization and tokenization; use template-supported assistant masks or verify token boundaries directly. Decode tokens with labels overlaid, test multi-turn conversations and empty replies, and count supervised tokens by role. A masking bug can train the model to reproduce prompts while reporting a reassuringly low loss.


In [ ]:
roles=["system","user","assistant","assistant","user","assistant"]; labels=[-100,-100,31,32,-100,44]
counts={r:sum(label!=-100 for rr,label in zip(roles,labels) if rr==r) for r in set(roles)}; print(counts); assert counts.get("user",0)==0


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [LoRA](https://arxiv.org/abs/2106.09685)
- [QLoRA](https://arxiv.org/abs/2305.14314)
- [PEFT documentation](https://huggingface.co/docs/peft/index)


## Exercises

    1. Create a 50-example format-learning dataset with a leakage-resistant split.
2. Compare target-module parameter counts for attention-only versus attention-plus-MLP.
3. Run a tiny adapter experiment and report base/adapter results with uncertainty.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
